In [1]:
print("Hello World!")

Hello World!


In [2]:
import requests
import pandas as pd
from defs import defs

In [3]:
session = requests.Session()

In [4]:
ins_df = pd.read_pickle("instruments.pkl")

In [5]:
our_curr = ['EUR', 'USD', 'GBP', 'JPY', 'CHF', 'NZD', 'CAD']

In [6]:
# ins_df.shape # num rows, num cols
# ins_df.info()
# ins_df.columns
# ins_df.name.unique()

In [7]:
def fetch_candles(pair_name, count, granularity):
    url = f"{defs.OANDA_URL}/instruments/{pair_name}/candles"
    params = dict(
        count = count,
        granularity = granularity,
        price = "MBA"
    )
    response = session.get(url, params=params, headers=defs.SECURE_HEADER)
    return response.status_code, response.json()


In [8]:
code, res = fetch_candles("EUR_USD", 10, "H1")

In [9]:
print(code)

200


In [10]:
print(res)

{'instrument': 'EUR_USD', 'granularity': 'H1', 'candles': [{'complete': True, 'volume': 7418, 'time': '2026-05-15T11:00:00.000000000Z', 'bid': {'o': '1.16484', 'h': '1.16493', 'l': '1.16285', 'c': '1.16310'}, 'mid': {'o': '1.16492', 'h': '1.16501', 'l': '1.16294', 'c': '1.16318'}, 'ask': {'o': '1.16500', 'h': '1.16509', 'l': '1.16301', 'c': '1.16325'}}, {'complete': True, 'volume': 7695, 'time': '2026-05-15T12:00:00.000000000Z', 'bid': {'o': '1.16308', 'h': '1.16314', 'l': '1.16181', 'c': '1.16255'}, 'mid': {'o': '1.16316', 'h': '1.16322', 'l': '1.16190', 'c': '1.16264'}, 'ask': {'o': '1.16324', 'h': '1.16330', 'l': '1.16197', 'c': '1.16272'}}, {'complete': True, 'volume': 11668, 'time': '2026-05-15T13:00:00.000000000Z', 'bid': {'o': '1.16256', 'h': '1.16316', 'l': '1.16183', 'c': '1.16229'}, 'mid': {'o': '1.16264', 'h': '1.16324', 'l': '1.16190', 'c': '1.16236'}, 'ask': {'o': '1.16272', 'h': '1.16331', 'l': '1.16198', 'c': '1.16243'}}, {'complete': True, 'volume': 11608, 'time': '2026

In [11]:
def get_candles_df(json_response):

    prices = ['mid', 'bid', 'ask']
    ohlc = ['o', 'h', 'l', 'c']
    
    our_data = []
    for candle in json_response['candles']:
        if candle['complete'] == False:
            continue
        new_dict = {}
        new_dict['time'] = candle['time']
        new_dict['volume'] = candle['volume']
        for price in prices:
            for oh in ohlc:
                new_dict[f"{price}_{oh}"] = candle[price][oh]
        our_data.append(new_dict)
    return pd.DataFrame.from_dict(our_data)

In [12]:
code, res = fetch_candles("EUR_USD", 10, "H1")

In [13]:
df = get_candles_df(res)

In [14]:
df.head()

,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2026-05-15T11:00:00.000000000Z,7418,1.16492,1.16501,1.16294,1.16318,1.16484,1.16493,1.16285,1.16310,1.16500,1.16509,1.16301,1.16325
1,2026-05-15T12:00:00.000000000Z,7695,1.16316,1.16322,1.16190,1.16264,1.16308,1.16314,1.16181,1.16255,1.16324,1.16330,1.16197,1.16272
2,2026-05-15T13:00:00.000000000Z,11668,1.16264,1.16324,1.16190,1.16236,1.16256,1.16316,1.16183,1.16229,1.16272,1.16331,1.16198,1.16243
3,2026-05-15T14:00:00.000000000Z,11608,1.16234,1.16379,1.16216,1.16294,1.16227,1.16370,1.16208,1.16287,1.16242,1.16388,1.16223,1.16302
4,2026-05-15T15:00:00.000000000Z,8551,1.16296,1.16311,1.16206,1.16265,1.16288,1.16303,1.16198,1.16256,1.16305,1.16319,1.16215,1.16274


In [15]:
def save_file(candles_df, pair, granularity):
    candles_df.to_pickle(f"his_data/{pair}_{granularity}.pkl")

In [16]:
def create_data(pair, granularity):
    code, json_data = fetch_candles(pair, 4000, granularity)
    if code != 200:
        print(pair, "Error")
        return
    df = get_candles_df(json_data)
    print(f"{pair} loaded {df.shape[0]} candles from {df.time.min()} to {df.time.max()}")
    save_file(df, pair, granularity)


In [17]:
for p1 in our_curr:
    for p2 in our_curr:
        pair = f"{p1}_{p2}"
        if pair in ins_df.name.unique():
            create_data(pair, "H1")

EUR_USD loaded 4000 candles from 2025-09-23T05:00:00.000000000Z to 2026-05-15T20:00:00.000000000Z
EUR_GBP loaded 4000 candles from 2025-09-23T05:00:00.000000000Z to 2026-05-15T20:00:00.000000000Z
EUR_JPY loaded 4000 candles from 2025-09-23T05:00:00.000000000Z to 2026-05-15T20:00:00.000000000Z
EUR_CHF loaded 4000 candles from 2025-09-23T05:00:00.000000000Z to 2026-05-15T20:00:00.000000000Z
EUR_NZD loaded 4000 candles from 2025-09-23T05:00:00.000000000Z to 2026-05-15T20:00:00.000000000Z
EUR_CAD loaded 4000 candles from 2025-09-23T05:00:00.000000000Z to 2026-05-15T20:00:00.000000000Z
USD_JPY loaded 4000 candles from 2025-09-23T05:00:00.000000000Z to 2026-05-15T20:00:00.000000000Z
USD_CHF loaded 4000 candles from 2025-09-23T05:00:00.000000000Z to 2026-05-15T20:00:00.000000000Z
USD_CAD loaded 4000 candles from 2025-09-23T05:00:00.000000000Z to 2026-05-15T20:00:00.000000000Z
GBP_USD loaded 4000 candles from 2025-09-23T05:00:00.000000000Z to 2026-05-15T20:00:00.000000000Z
GBP_JPY loaded 4000 